# Canada Immigration Analysis - Data Cleaning & EDA
Three-layer analytical chain: Selection (who got invited) → Flow (who landed) → Outcome (income).
Data sources documented in `data/SOURCES.md`. All raw files are Tab-separated, UTF-8 encoded.

## 1. Layer 1 Selection:  EE invitation data (By Province/Category)
source: `ee_invitations_by_province_category.csv`
Cleaning goals: convert suppression symbol `--` in `TOTAL` to numeric; drop redundant French columns.
Note: excludes category-based selection (CBS) draws introduced in 2023.

In [1]:
import pandas as pd
df = pd.read_csv("../data/raw/ee_invitations_by_province_category.csv",sep="\t",encoding="utf-8")
df.head()
# df.info()

,EN_YEAR,FR_ANNEÉ,EN_PROVINCE_TERRITORY,FR_PROVINCE_TERRITOIRE,EN_INVITATION_CATEGORY,FR_CATEGORIE_D'INVITATION,TOTAL
0,2015,2015,Alberta,Alberta,Canadian Experience Class,Catégorie de l'expérience canadienne,4635
1,2015,2015,Alberta,Alberta,Federal Skilled Trades,Métiers spécialisés,1385
2,2015,2015,Alberta,Alberta,Federal Skilled Worker,Travailleurs qualifiés,1970
3,2015,2015,British Columbia,Colombie-Britannique,Canadian Experience Class,Catégorie de l'expérience canadienne,1295
4,2015,2015,British Columbia,Colombie-Britannique,Federal Skilled Trades,Métiers spécialisés,200


In [2]:
#
# df[~df['TOTAL'].str.isdigit()]['TOTAL'].unique()
df['TOTAL'] = pd.to_numeric(df['TOTAL'],errors='coerce')

In [3]:
df = df.drop(columns=["FR_ANNEÉ", "FR_PROVINCE_TERRITOIRE", "FR_CATEGORIE_D'INVITATION"])
df.head()

,EN_YEAR,EN_PROVINCE_TERRITORY,EN_INVITATION_CATEGORY,TOTAL
0,2015,Alberta,Canadian Experience Class,4635.0
1,2015,Alberta,Federal Skilled Trades,1385.0
2,2015,Alberta,Federal Skilled Worker,1970.0
3,2015,British Columbia,Canadian Experience Class,1295.0
4,2015,British Columbia,Federal Skilled Trades,200.0


In [4]:
print("province/territory: \n", df["EN_PROVINCE_TERRITORY"].unique())
print("invitation category: \n", df["EN_INVITATION_CATEGORY"].unique())
print("year: \n",df["EN_YEAR"].min(), df["EN_YEAR"].max())


province/territory: 
 <StringArray>
[                      'Alberta',              'British Columbia',
                      'Manitoba',                 'New Brunswick',
     'Newfoundland and Labrador',         'Northwest Territories',
                   'Nova Scotia',                       'Nunavut',
                       'Ontario',          'Prince Edward Island',
 'Province/Territory not stated',                  'Saskatchewan',
                         'Yukon']
Length: 13, dtype: str
invitation category: 
 <StringArray>
[ 'Canadian Experience Class',     'Federal Skilled Trades',
     'Federal Skilled Worker', 'Provincial Nominee Program',
                 'Not Stated']
Length: 5, dtype: str
year: 
 2015 2026


In [5]:
# df.info()
print(df[df["EN_PROVINCE_TERRITORY"] == "Province/Territory not stated"]["TOTAL"].sum())
print(df[df["EN_INVITATION_CATEGORY"] == "Not Stated"]["TOTAL"].sum())
print(df["TOTAL"].sum())


0.0
0.0
766380.0


In [6]:
df.groupby("EN_YEAR")["TOTAL"].sum()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 471 entries, 0 to 470
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   EN_YEAR                 471 non-null    int64  
 1   EN_PROVINCE_TERRITORY   471 non-null    str    
 2   EN_INVITATION_CATEGORY  471 non-null    str    
 3   TOTAL                   422 non-null    float64
dtypes: float64(1), int64(1), str(2)
memory usage: 14.8 KB


In [7]:
df.to_csv("../data/processed/ee_invitations_cleaned.csv", index=False)


## 2. Layer 1 slection: EE invitation data - by score range
source: `ee_ita_score_breakdown.csv`
Adds a CRS/ITA score-band dimension to ①, used to analyze selection thresholds and score distribution.
Cleaning goals: handle `--`; drop French columns. More suppression expected after score-band split.

In [8]:
df_ita = pd.read_csv('../data/raw/ee_ita_score_breakdown.csv', sep='\t', encoding='utf-8')
df_ita.head()
df_ita.info()

<class 'pandas.DataFrame'>
RangeIndex: 1912 entries, 0 to 1911
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   EN_YEAR                    1912 non-null   int64
 1   FR_ANNEÉ                   1912 non-null   int64
 2   EN_PROVINCE_TERRITORY      1912 non-null   str  
 3   FR_PROVINCE_TERRITOIRE     1912 non-null   str  
 4   EN_INVITATION_CATEGORY     1912 non-null   str  
 5   FR_CATEGORIE_D'INVITATION  1912 non-null   str  
 6   EN_ITA_SCORE               1912 non-null   str  
 7   FR_NOTE_D'IPD              1912 non-null   str  
 8   TOTAL                      1912 non-null   str  
dtypes: int64(2), str(7)
memory usage: 134.6 KB


In [9]:
df_ita[~df_ita["TOTAL"].str.isdigit()]["TOTAL"].unique()
df_ita['TOTAL'] = pd.to_numeric(df_ita['TOTAL'],errors="coerce")

In [10]:
df_ita = df_ita.drop(columns=['FR_ANNEÉ','FR_PROVINCE_TERRITOIRE',"FR_CATEGORIE_D'INVITATION","FR_NOTE_D'IPD" ])

In [11]:
df_ita.info()
df_ita.to_csv("../data/processed/ee_ita_score_cleaned.csv", index=False)


<class 'pandas.DataFrame'>
RangeIndex: 1912 entries, 0 to 1911
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   EN_YEAR                 1912 non-null   int64  
 1   EN_PROVINCE_TERRITORY   1912 non-null   str    
 2   EN_INVITATION_CATEGORY  1912 non-null   str    
 3   EN_ITA_SCORE            1912 non-null   str    
 4   TOTAL                   1313 non-null   float64
dtypes: float64(1), int64(1), str(3)
memory usage: 74.8 KB


## 3. Layer 2 — Flow: PR admissions (by province/immigration category)
Source: `pr_admissions_by_province_immcat.csv`
Actual permanent-resident landings, at month × province × category granularity.
Cleaning goals: handle `--`; drop French columns. Note: category names differ from ① (need mapping).

In [12]:
df_pr_cat = pd.read_csv("../data/raw/pr_admissions_by_province_immcat.csv", sep="\t", encoding="utf-8")
df_pr_cat.info()

<class 'pandas.DataFrame'>
RangeIndex: 4168 entries, 0 to 4167
Data columns (total 11 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   EN_YEAR                                4168 non-null   int64
 1   EN_QUARTER                             4168 non-null   str  
 2   EN_MONTH                               4168 non-null   str  
 3   FR_ANNEÉ                               4168 non-null   int64
 4   FR_TRIMESTRE                           4168 non-null   str  
 5   FR_MOIS                                4168 non-null   str  
 6   EN_PROVINCE_TERRITORY                  4168 non-null   str  
 7   FR_PROVINCE_TERRITOIRE                 4168 non-null   str  
 8   EN_IMMIGRATION_CATEGORY-COMPONENT      4168 non-null   str  
 9   FR_CATÉGORIE_D'IMMIGRATION-COMPOSANTE  4168 non-null   str  
 10  TOTAL                                  4168 non-null   str  
dtypes: int64(2), str(9)
memory usage: 358.3 K

In [13]:
df_pr_cat[~df_pr_cat["TOTAL"].str.isdigit()]["TOTAL"].unique()
df_pr_cat['TOTAL'] = pd.to_numeric(df_pr_cat['TOTAL'],errors='coerce')

In [14]:
df_pr_cat = df_pr_cat.drop(columns=["FR_ANNEÉ", "FR_TRIMESTRE", "FR_MOIS", "FR_PROVINCE_TERRITOIRE", "FR_CATÉGORIE_D'IMMIGRATION-COMPOSANTE"])

In [15]:
df_pr_cat.info()

<class 'pandas.DataFrame'>
RangeIndex: 4168 entries, 0 to 4167
Data columns (total 6 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   EN_YEAR                            4168 non-null   int64  
 1   EN_QUARTER                         4168 non-null   str    
 2   EN_MONTH                           4168 non-null   str    
 3   EN_PROVINCE_TERRITORY              4168 non-null   str    
 4   EN_IMMIGRATION_CATEGORY-COMPONENT  4168 non-null   str    
 5   TOTAL                              3246 non-null   float64
dtypes: float64(1), int64(1), str(4)
memory usage: 195.5 KB


In [16]:
df_pr_cat["EN_IMMIGRATION_CATEGORY-COMPONENT"].unique()

<StringArray>
[       'Canadian Experience',              'Skilled Trade',
 'Provincial Nominee Program',             'Skilled Worker']
Length: 4, dtype: str

In [17]:
# the category name in this file is different from others, so we mapping
category_map = {
    "Canadian Experience": "Canadian Experience Class",
    "Skilled Trade": "Federal Skilled Trades",
    "Skilled Worker": "Federal Skilled Worker",
    "Provincial Nominee Program": "Provincial Nominee Program",
}
df_pr_cat["category_standard"] = df_pr_cat["EN_IMMIGRATION_CATEGORY-COMPONENT"].map(category_map)
df_pr_cat.head()

,EN_YEAR,EN_QUARTER,EN_MONTH,EN_PROVINCE_TERRITORY,EN_IMMIGRATION_CATEGORY-COMPONENT,TOTAL,category_standard
0,2025,Q4,Dec,Ontario,Canadian Experience,150.0,Canadian Experience Class
1,2016,Q1,Jan,Alberta,Skilled Trade,275.0,Federal Skilled Trades
2,2025,Q3,Jul,Manitoba,Canadian Experience,35.0,Canadian Experience Class
3,2016,Q3,Sep,Nova Scotia,Provincial Nominee Program,115.0,Provincial Nominee Program
4,2019,Q1,Feb,Alberta,Provincial Nominee Program,10.0,Provincial Nominee Program


In [18]:
df_pr_cat.to_csv("../data/processed/pr_admissions_by_category_cleaned.csv", index=False)

## 4. Layer 2 — Flow: PR admissions (by country of citizenship)

Source: `pr_admissions_by_citizenship.csv`
Source-country distribution, at month × province × citizenship granularity (~200 countries, largest file).
Cleaning goals: handle `--`; drop French columns. No category dimension — parallel view to ③.


In [19]:
df_pr_citz = pd.read_csv("../data/raw/pr_admissions_by_citizenship.csv", sep="\t", encoding="utf-8")
df_pr_citz.info()

<class 'pandas.DataFrame'>
RangeIndex: 45453 entries, 0 to 45452
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   EN_YEAR                    45453 non-null  int64
 1   EN_QUARTER                 45453 non-null  str  
 2   EN_MONTH                   45453 non-null  str  
 3   FR_ANNEÉ                   45453 non-null  int64
 4   FR_TRIMESTRE               45453 non-null  str  
 5   FR_MOIS                    45453 non-null  str  
 6   EN_PROVINCE_TERRITORY      45453 non-null  str  
 7   FR_PROVINCE_TERRITOIRE     45453 non-null  str  
 8   EN_COUNTRY_OF_CITIZENSHIP  45453 non-null  str  
 9   FR_PAYS_DE_CITOYENNETÉ     45453 non-null  str  
 10  TOTAL                      45453 non-null  str  
dtypes: int64(2), str(9)
memory usage: 3.8 MB


In [20]:
df_pr_citz[~df_pr_citz["TOTAL"].str.isdigit()]["TOTAL"].unique()
df_pr_citz["TOTAL"] = pd.to_numeric(df_pr_citz["TOTAL"], errors="coerce")

In [21]:
df_pr_citz = df_pr_citz.drop(columns=["FR_ANNEÉ", "FR_TRIMESTRE", "FR_MOIS", "FR_PROVINCE_TERRITOIRE", "FR_PAYS_DE_CITOYENNETÉ"])

In [22]:
df_pr_citz.info()
df_pr_citz.to_csv("../data/processed/pr_admissions_by_citizenship_cleaned.csv", index=False)


<class 'pandas.DataFrame'>
RangeIndex: 45453 entries, 0 to 45452
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   EN_YEAR                    45453 non-null  int64  
 1   EN_QUARTER                 45453 non-null  str    
 2   EN_MONTH                   45453 non-null  str    
 3   EN_PROVINCE_TERRITORY      45453 non-null  str    
 4   EN_COUNTRY_OF_CITIZENSHIP  45453 non-null  str    
 5   TOTAL                      19559 non-null  float64
dtypes: float64(1), int64(1), str(4)
memory usage: 2.1 MB


## 5. Layer 3 — Outcome： StatCan immigrant income (by admission cohort)

Source: `statcan_immigrant_income_by_cohort.csv`
Already produced in clean long format by `download_statcan_income.py` — minimal cleaning needed.
Dimensions: admission cohort × category × income statistic × tax year. Income data lags to 2023.


In [ ]:
df_income = pd.read_csv("../data/raw/statcan_immigrant_income_by_cohort.csv")
df_income.info()

<class 'pandas.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   admission_cohort_year  396 non-null    int64  
 1   immigrant_category     396 non-null    str    
 2   statistic              396 non-null    str    
 3   tax_year               396 non-null    int64  
 4   value                  396 non-null    float64
dtypes: float64(1), int64(2), str(2)
memory usage: 15.6 KB


In [ ]:
df_income.to_csv("../data/processed/statcan_immigrant_income_by_cohort.csv", index=False)